In [2]:
import pandas as pd

df = pd.read_csv('2019-Oct.csv')

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [3]:
smartphone_df = df[df['category_code'] == 'electronics.smartphone'].copy()

In [4]:
#정렬 + unknown처리
df = smartphone_df.sort_values(['user_session', 'event_time']).copy()
df['brand'] = df['brand'].fillna('unknown')

In [5]:
grouped = df.groupby(['user_session', 'product_id'])

In [6]:
event_first_time = (
    smartphone_df
    .groupby(['user_session', 'product_id', 'event_type'])['event_time']
    .min()
    .unstack()
    .reset_index()
)

for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time.columns:
        event_first_time[col] = pd.NaT

# 존재 여부 컬럼 따로 생성
event_first_time['has_view'] = event_first_time['view'].notna()
event_first_time['has_cart'] = event_first_time['cart'].notna()
event_first_time['has_purchase'] = event_first_time['purchase'].notna()

# 순서 검증
event_first_time['view_to_cart'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    (event_first_time['view'] < event_first_time['cart'])
)

event_first_time['view_to_cart_to_purchase'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    event_first_time['has_purchase'] &
    (event_first_time['view'] < event_first_time['cart']) &
    (event_first_time['cart'] < event_first_time['purchase'])
)

funnel_df = event_first_time[[
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

In [7]:
funnel_summary = pd.DataFrame({
    'step': ['view', 'view_to_cart', 'view_to_cart_to_purchase'],
    'count': [
        funnel_df['view'].sum(),
        funnel_df['view_to_cart'].sum(),
        funnel_df['view_to_cart_to_purchase'].sum()
    ]
})

funnel_summary['conversion_rate_vs_view'] = (
    funnel_summary['count'] / funnel_summary.loc[0, 'count'] * 100
)

funnel_summary

,step,count,conversion_rate_vs_view
0,view,7058659,100.000000
1,view_to_cart,364365,5.161958
2,view_to_cart_to_purchase,191430,2.711988


In [8]:
view_sessions = funnel_df['view'].sum()
view_to_cart_sessions = funnel_df['view_to_cart'].sum()
view_to_cart_to_purchase_sessions = funnel_df['view_to_cart_to_purchase'].sum()

view_to_cart_rate = view_to_cart_sessions / view_sessions
cart_to_purchase_rate = view_to_cart_to_purchase_sessions / view_to_cart_sessions
total_conversion_rate = view_to_cart_to_purchase_sessions / view_sessions

In [10]:
target_brands = ['samsung', 'apple', 'xiaomi']

df_flow = df[df['brand'].isin(target_brands)].copy()
df_flow = df_flow.sort_values(['user_session', 'event_time'])

flow_df = (
    df_flow[df_flow['event_type'].isin(['view', 'cart', 'purchase'])]
    .groupby(['user_session', 'event_type'])['brand']
    .first()
    .unstack()
    .reset_index()
)

flow_df = flow_df.rename(columns={
    'view': 'view_brand',
    'cart': 'cart_brand',
    'purchase': 'purchase_brand'
})

flow_df.head()

event_type,user_session,cart_brand,purchase_brand,view_brand
0,00000056-a206-40dd-b174-a072550fa38c,NaN,NaN,apple
1,00000083-8816-4d58-a9b8-f52f54186edc,samsung,samsung,samsung
2,000001fd-1f89-45e8-a3ce-fe3218cabfad,samsung,samsung,samsung
3,00000a05-fa4e-4486-b5e8-c8bffa63b422,NaN,NaN,samsung
4,00000aaa-d774-49bc-9c31-0c9f6e1c2f0a,NaN,NaN,xiaomi


In [11]:
def classify_flow(row):
    v = row.get('view_brand')
    c = row.get('cart_brand')
    p = row.get('purchase_brand')

    brands = [x for x in [v, c, p] if pd.notna(x)]

    if pd.isna(v) or pd.isna(p):
        return 'no_purchase_or_no_view'

    if pd.notna(c):
        if v == c == p:
            return 'same_brand'
        elif v != c and c == p:
            return 'view_to_cart_switch'
        elif v == c and c != p:
            return 'cart_to_purchase_switch'
        elif len(set(brands)) == 3:
            return 'multi_brand_exploration'
        else:
            return 'etc'
    else:
        if v == p:
            return 'direct_same_purchase'
        else:
            return 'direct_switch_purchase'

flow_df['pattern'] = flow_df.apply(classify_flow, axis=1)

flow_df['pattern'].value_counts()

pattern
no_purchase_or_no_view     2488727
same_brand                  156658
direct_same_purchase         83699
view_to_cart_switch           6148
direct_switch_purchase        3072
cart_to_purchase_switch        848
etc                            661
multi_brand_exploration         38
Name: count, dtype: int64

In [12]:
flow_df['pattern'].value_counts(normalize=True) * 100

pattern
no_purchase_or_no_view     90.834392
same_brand                  5.717756
direct_same_purchase        3.054874
view_to_cart_switch         0.224392
direct_switch_purchase      0.112123
cart_to_purchase_switch     0.030951
etc                         0.024125
multi_brand_exploration     0.001387
Name: proportion, dtype: float64

view → cart 변경 분석

In [13]:
vc_switch = flow_df[flow_df['pattern'] == 'view_to_cart_switch'].copy()

vc_matrix = (
    vc_switch.groupby(['view_brand', 'cart_brand'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

vc_matrix

,view_brand,cart_brand,count
0,apple,samsung,1838
3,samsung,xiaomi,1587
2,samsung,apple,1454
5,xiaomi,samsung,758
1,apple,xiaomi,356
4,xiaomi,apple,155


cart → purchase 변경 분석

In [14]:
cp_switch = flow_df[flow_df['pattern'] == 'cart_to_purchase_switch'].copy()

cp_matrix = (
    cp_switch.groupby(['cart_brand', 'purchase_brand'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

cp_matrix

,cart_brand,purchase_brand,count
0,apple,samsung,295
3,samsung,xiaomi,203
2,samsung,apple,199
5,xiaomi,samsung,103
1,apple,xiaomi,35
4,xiaomi,apple,13


direct 구매 변경 (view → purchase)

In [15]:
dp_switch = flow_df[flow_df['pattern'] == 'direct_switch_purchase'].copy()

dp_matrix = (
    dp_switch.groupby(['view_brand', 'purchase_brand'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

dp_matrix

,view_brand,purchase_brand,count
3,samsung,xiaomi,966
0,apple,samsung,769
2,samsung,apple,711
5,xiaomi,samsung,300
1,apple,xiaomi,229
4,xiaomi,apple,97


In [16]:
vc_ratio = vc_matrix.copy()

vc_ratio['total_from_view'] = vc_ratio.groupby('view_brand')['count'].transform('sum')
vc_ratio['ratio'] = vc_ratio['count'] / vc_ratio['total_from_view'] * 100

vc_ratio.sort_values(['view_brand', 'ratio'], ascending=[True, False])

,view_brand,cart_brand,count,total_from_view,ratio
0,apple,samsung,1838,2194,83.773929
1,apple,xiaomi,356,2194,16.226071
3,samsung,xiaomi,1587,3041,52.186781
2,samsung,apple,1454,3041,47.813219
5,xiaomi,samsung,758,913,83.023001
4,xiaomi,apple,155,913,16.976999


In [18]:
cp_ratio = cp_matrix.copy()

cp_ratio['total_from_cart'] = cp_ratio.groupby('cart_brand')['count'].transform('sum')
cp_ratio['ratio'] = cp_ratio['count'] / cp_ratio['total_from_cart'] * 100

cp_ratio.sort_values(['cart_brand', 'ratio'], ascending=[True, False])

,cart_brand,purchase_brand,count,total_from_cart,ratio
0,apple,samsung,295,330,89.393939
1,apple,xiaomi,35,330,10.606061
3,samsung,xiaomi,203,402,50.497512
2,samsung,apple,199,402,49.502488
5,xiaomi,samsung,103,116,88.793103
4,xiaomi,apple,13,116,11.206897


In [19]:
dp_ratio = dp_matrix.copy()

dp_ratio['total_from_view'] = dp_ratio.groupby('view_brand')['count'].transform('sum')
dp_ratio['ratio'] = dp_ratio['count'] / dp_ratio['total_from_view'] * 100

dp_ratio.sort_values(['view_brand', 'ratio'], ascending=[True, False])

,view_brand,purchase_brand,count,total_from_view,ratio
0,apple,samsung,769,998,77.054108
1,apple,xiaomi,229,998,22.945892
3,samsung,xiaomi,966,1677,57.602862
2,samsung,apple,711,1677,42.397138
5,xiaomi,samsung,300,397,75.566751
4,xiaomi,apple,97,397,24.433249
